# LLM-based Traveller Persona Generation

This notebook shows how personas are generated from a travel survey with an LLM. It has three parts:

1. Prepare the data for generation.
2. Use an LLM to generate one persona for each person.
3. Remove highly similar personas and keep a compact persona set.


In [ ]:
import json
import math
import os
import re
import time
from typing import Dict

import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm

# 1. Data preparation

This work uses three datasets from the National Travel Survey: household, individual, and trip. They were merged, keeping 2021 and the variables needed here. To limit storage, the original three releases are not included, and this folder starts from the processed data.

That processed dataset is provided as two files. `trip_indi_hh_2021.csv` contains the variables and their numeric codes. `lookup_table_response.xlsx` gives the label for each code.

The purpose of data preparation is to turn the survey table into records an LLM can read. Variable names and numeric codes are replaced with plain English. Missing values are dropped, and some age and mode values are grouped into broader categories. The table is then turned into one JSON record per person, with individual attributes, household attributes, and trips grouped by day

In [ ]:
survey_path = "../data/trip_indi_hh_2021.csv"
lookup_path = "../data/lookup_table_response.xlsx"

# keys mapped to plain English
INDIV_KEY_MAP: Dict[str, str] = {
    "Age_B04ID": "age band",
    "WkPlace_B01ID": "workplace",
    "EcoStat_B02ID": "working status",
    "OrdBus2Freq_B01ID": "bus usage frequency",
    "TaxiCab2Freq_B01ID": "taxi usage frequency",
    "Bicycle3Freq_B01ID": "bicycle usage frequency",
    "Walk2Freq_B01ID": "walking frequency",
    "PrivCar2_B01ID": "private car usage frequency",
    "WkMode_B01ID": "usual work travel mode",
    "SchTrav_B01ID": "usual school travel mode",
}

HH_KEY_MAP: Dict[str, str] = {
    "HHIncQDS2021Eng_B01ID": "household income quintile",
}

TRIP_KEY_MAP: Dict[str, str] = {
    "JourSeq": "journey sequence",
    "MainMode_B04ID": "trip mode",
    "TripPurpFrom_B01ID": "purpose of the origin",
    "TripPurpTo_B01ID": "purpose of the destination",
    "start_time_hm": "start time",
    "end_time_hm": "end time",
    "TripDisIncSW": "trip distance (miles)",
    "TripTotalTime": "trip duration (minutes)",
}

# special-case codes 
## missing values
SKIP_CODES = {-8, -9, -10}

## broad categories
WKMODE_SPECIAL_MAP = {
    "1": "Car / van",  # Car / van - driver
    "2": "Car / van",  # Car / van - passenger
    "3": "Car / van",  # Car / van - either driver or passenger
    "4": "Car / van",  # Car / van - no driver/passenger details
    "9": "Underground / metro / light rail / tram",  # Underground / metro / light rail / tram (excludes Light rail prior to 2002)
    "10": "Underground / metro / light rail / tram",  # Light rail (1998-2001 only)
    "13": "Taxi / minicab",  # Taxi / minicab (2002 onwards)
}

SCHTRAV_SPECIAL_MAP = {
    "3": "Bus / minibus / coach",  # Public bus, minibus or coach
    "4": "Bus / minibus / coach",  # School or LA bus, minibus or coach
}

MAINMODE_SPECIAL_MAP = {
    "3": "Car / van",  # Car / van driver
    "4": "Car / van",  # Car / van passenger
    "7": "Bus",        # Bus in London
    "8": "Bus",        # Other local bus
    "9": "Bus",        # Non-local bus
}

AGE_BAND_MAP = {
    "1": "children and teenagers",  # 0 - 4 years
    "2": "children and teenagers",  # 5 - 10 years
    "3": "children and teenagers",  # 11 - 16 years
    "4": "young adults",            # 17 - 20 years
    "5": "young adults",            # 21 - 29 years
    "6": "middle-aged adults",      # 30 - 39 years
    "7": "middle-aged adults",      # 40 - 49 years
    "8": "middle-aged adults",      # 50 - 59 years
    "9": "seniors",                 # 60 years +
}

SPECIAL_LABEL_MAPS = {
    "WkMode_B01ID": WKMODE_SPECIAL_MAP,
    "SchTrav_B01ID": SCHTRAV_SPECIAL_MAP,
    "MainMode_B04ID": MAINMODE_SPECIAL_MAP,
    "Age_B04ID": AGE_BAND_MAP,
}


def code_key(value):
    """Turn a code into the lookup string. 1.0 becomes "1"."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, float) and value.is_integer():
        return str(int(value))
    return str(value)

def to_json_value(value):
    """Convert numpy values to plain Python so a record can be stored as JSON."""
    if isinstance(value, dict):
        return {str(k): to_json_value(v) for k, v in value.items()}
    elif isinstance(value, list):
        return [to_json_value(v) for v in value]
    elif isinstance(value, (np.integer,)):
        return int(value)
    elif isinstance(value, (np.floating,)):
        number = float(value)
        return int(number) if number.is_integer() else number
    elif isinstance(value, (np.bool_,)):
        return bool(value)
    elif pd.isna(value):
        return None
    else:
        return value

def load_lookup_maps(lookup_path: str) -> Dict[str, Dict[str, str]]:
    lk = pd.read_excel(lookup_path, sheet_name=0)
    lk.columns = [str(c) for c in lk.columns]
    maps: Dict[str, Dict[str, str]] = {}
    for var, sub in lk.groupby("Variable", sort=False):
        m = {}
        for _, r in sub.iterrows():
            code = code_key(r.get("ID"))
            label = r.get("Desc")
            if code is not None:
                m[code] = None if (pd.isna(label)) else str(label)
        maps[str(var)] = m
    return maps

def should_skip_code(code_val):
    """True for the missing values -8, -9, and -10."""
    if code_val is None or (isinstance(code_val, float) and math.isnan(code_val)):
        return False
    try:
        return int(float(code_val)) in SKIP_CODES
    except (ValueError, TypeError):
        return False

def code_label(row: pd.Series, col: str, maps: Dict[str, Dict[str, str]]):
    """Return the text for a code. An empty cell returns None."""
    code_val = row.get(col)
    code_str = code_key(code_val)
    special = SPECIAL_LABEL_MAPS.get(col, {})
    if code_str in special:
        return special[code_str]
    label = maps.get(col, {}).get(code_str)
    if label is not None:
        return label
    if code_str is None:
        return None
    return str(code_val)

def safe_numeric_convert(value, target_type=float):
    if pd.isna(value):
        return None
    try:
        if target_type == int:
            return int(float(value))
        else:
            return float(value)
    except (ValueError, TypeError):
        return None

def field_labels(row, key_map, maps):
    """Map survey columns to text labels. Skip missing columns and missing values."""
    fields = {}
    for col, key in key_map.items():
        if col not in row.index or should_skip_code(row.get(col)):
            continue
        fields[key] = code_label(row, col, maps)
    return fields

def load_nts_records(survey_path: str, lookup_path: str):
    df = pd.read_csv(survey_path)
    maps = load_lookup_maps(lookup_path)

    sort_cols = [c for c in ["IndividualID", "TravDay", "JourSeq"] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols)

    records = []
    for individual_id, person_rows in df.groupby("IndividualID", dropna=False):
        first_row = person_rows.iloc[0]
        individual = field_labels(first_row, INDIV_KEY_MAP, maps)
        household = field_labels(first_row, HH_KEY_MAP, maps)

        days = []
        for day_val, day_rows in person_rows.groupby("TravDay", dropna=False):
            if isinstance(day_val, float) and math.isnan(day_val):
                continue
            day_item = {"day": day_val, "trips": []}

            for _, row in day_rows.sort_values("JourSeq").iterrows():
                trip = {}
                if "JourSeq" in row.index:
                    trip[TRIP_KEY_MAP["JourSeq"]] = row["JourSeq"]
                trip.update(field_labels(row, {
                    col: TRIP_KEY_MAP[col]
                    for col in ["MainMode_B04ID", "TripPurpFrom_B01ID", "TripPurpTo_B01ID"]
                }, maps))
                for col in ["start_time_hm", "end_time_hm"]:
                    trip[TRIP_KEY_MAP[col]] = row.get(col) if col in row.index and pd.notna(row.get(col)) else None
                if "TripDisIncSW" in row.index:
                    trip[TRIP_KEY_MAP["TripDisIncSW"]] = safe_numeric_convert(row.get("TripDisIncSW"), float)
                if "TripTotalTime" in row.index:
                    trip[TRIP_KEY_MAP["TripTotalTime"]] = safe_numeric_convert(row.get("TripTotalTime"), int)
                day_item["trips"].append(trip)

            days.append(day_item)

        record = {
            "individual_id": individual_id,
            "household_id": first_row.get("HouseholdID") if "HouseholdID" in first_row else None,
            "individual": individual or None,
            "household": household or None,
            "trip": days,
        }
        records.append(to_json_value(record))
    return records

# run
nts_records = load_nts_records(survey_path, lookup_path)

# 2. Generate personas

Ask an LLM to infer one persona per record. Append each generated persona to `persona_initial.jsonl`.


In [ ]:
def request_persona(client, record, prompt_template, system_message, config):
    """Ask an LLM for one person's persona."""
    prompt = prompt_template.replace(
        "{json_str}",
        json.dumps(record, indent=2, ensure_ascii=False),
    )
    try:
        response = client.chat.completions.create(
            model=config["llm_model"],
            messages=[
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt},
            ],
            temperature=config["temperature"],
            max_tokens=config["max_tokens"],
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Generation failed: {e}"


def write_persona_file(records, prompt_path, output_jsonl, system_message, config):
    """Append successful personas to a JSONL file.

    Re-running skips IDs already written. A failed call is retried twice,
    then left unwritten so a later run can try again.
    """
    with open(prompt_path, encoding="utf-8") as f:
        prompt_template = f.read().strip()

    client = OpenAI(api_key=config["api_key"])

    def failed(persona):
        return isinstance(persona, str) and persona.startswith("Generation failed")

    processed_ids = set()
    if os.path.exists(output_jsonl):
        with open(output_jsonl, encoding="utf-8") as f:
            for line in f:
                try:
                    data = json.loads(line.strip())
                except Exception:
                    continue
                if "individual_id" not in data or failed(data.get("persona")):
                    continue
                processed_ids.add(data["individual_id"])

    total = len(records)
    succeeded = sum(1 for record in records if record.get("individual_id") in processed_ids)
    print("Persona generation started")
    progress = tqdm(total=total, initial=succeeded, bar_format="Progress: {n_fmt}/{total_fmt}")
    attempts = 0
    with open(output_jsonl, "a", encoding="utf-8") as outfile:
        for record in records:
            individual_id = record.get("individual_id")
            if individual_id in processed_ids:
                continue

            persona = None
            for attempt in range(1, 4):
                persona = request_persona(
                    client, record, prompt_template, system_message, config
                )
                attempts += 1
                if failed(persona) and attempt == 3:
                    tqdm.write(f"Generation failed for {individual_id}: {persona}")
                if attempts % 30 == 0:
                    time.sleep(60)
                else:
                    time.sleep(2)
                if not failed(persona):
                    break
            else:
                continue

            outfile.write(
                json.dumps(
                    {"individual_id": individual_id, "persona": persona},
                    ensure_ascii=False,
                )
                + "\n"
            )
            outfile.flush()
            processed_ids.add(individual_id)
            progress.update(1)

    progress.close()


with open("config.json", encoding="utf-8") as f:
    config = json.load(f)
with open("prompts/system.txt", encoding="utf-8") as f:
    system_message = f.read().strip()

output_jsonl = "../data/persona_initial.jsonl"
write_persona_file(
    nts_records,
    "prompts/nts_persona.txt",
    output_jsonl,
    system_message,
    config,
)


# 3. Deduplication

Remove near-duplicate personas in two steps. MinHash runs first. Embedding then compares the sentences that MinHash kept. This writes `minhash_report.txt`, `embedding_report.txt`, and `persona_final.txt`.


## 3.1 MinHash

Drop similar personas based on lexical similarity. 

In [ ]:
with open("config.json", encoding="utf-8") as f:
    config = json.load(f)

minhash_report = "../data/minhash_report.txt"
MINHASH_THRESHOLD = config["minhash_threshold"]

def preprocess_text(text):
    """Normalize English text before hashing."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text.strip())
    # Keep letters and digits, drop punctuation
    tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text)
    return tokens

def create_minhash(text, num_perm=128):
    """Build a MinHash signature."""
    m = MinHash(num_perm=num_perm)
    tokens = preprocess_text(text)
    for token in tokens:
        m.update(token.encode('utf8'))
    return m

def deduplicate_by_minhash():
    input_jsonl = "../data/persona_initial.jsonl"
    personas = []
    with open(input_jsonl, encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            persona = json.loads(line).get("persona", "").strip()
            if persona:
                personas.append(persona)
    
    print(f"Read {len(personas)} personas")
    
    lsh = MinHashLSH(threshold=MINHASH_THRESHOLD, num_perm=128)
    unique_personas = []
    removed_personas = []
    
    for idx, text in enumerate(personas):
        mh = create_minhash(text)
        
        similar_keys = lsh.query(mh)
        
        if similar_keys:
            similar_idx = int(similar_keys[0][1:])  # Index encoded in the key, e.g. p123
            removed_personas.append({
                'removed': text,
                'removed_idx': idx,
                'similar_to': personas[similar_idx],
                'similar_idx': similar_idx
            })
        else:
            key = f"p{idx}"
            lsh.insert(key, mh)
            unique_personas.append(text)
    
    print(f"{len(unique_personas)} personas remain after MinHash")
    print(f"Removed {len(removed_personas)} duplicates")
    
    with open(minhash_report, "w", encoding="utf-8") as f:
        f.write("MinHash deduplication report\n")
        f.write(f"Threshold: {MINHASH_THRESHOLD}\n")
        f.write(f"Input count: {len(personas)}\n")
        f.write(f"Kept: {len(unique_personas)}\n")
        f.write(f"Removed: {len(removed_personas)}\n")
        f.write(f"Removal rate: {len(removed_personas)/len(personas)*100:.2f}%\n")
        f.write("="*80 + "\n\n")
        
        for i, item in enumerate(removed_personas, 1):
            f.write(f"Duplicate pair {i}:\n")
            f.write(f"Kept (index {item['similar_idx']}): {item['similar_to']}\n")
            f.write(f"Removed (index {item['removed_idx']}): {item['removed']}\n")
            f.write("-" * 80 + "\n\n")
    
    print(f"Saved report to {minhash_report}")
    return unique_personas

minhash_personas = deduplicate_by_minhash()


## 3.2 Embedding

Drop similar personas based on semantic similarity. 


In [ ]:
with open("config.json", encoding="utf-8") as f:
    config = json.load(f)

persona_final = "../data/persona_final.txt"
embedding_report = "../data/embedding_report.txt"
EMBEDDING_THRESHOLD = config["embedding_threshold"]

def get_openai_embeddings(texts, model, batch_size=100):
    """Fetch embeddings from the OpenAI API."""
    client = OpenAI(api_key=config["api_key"])
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(
            model=model,
            input=batch,
        )
        embeddings.extend(item.embedding for item in response.data)
        time.sleep(0.1)  # Stay under the API rate limit
    return np.array(embeddings)

def get_fallback_embeddings(texts):
    """Use sentence-transformers when the OpenAI API is unavailable. This is not the paper's model."""
    print("Using sentence-transformers as the fallback...")
    os.environ["USE_TF"] = "0"
    os.environ["TRANSFORMERS_NO_TF"] = "1"
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return model.encode(
        texts,
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
    )

def deduplicate_by_embedding():
    personas = minhash_personas
    print(f"Read {len(personas)} personas after MinHash")

    print("Fetching text embeddings...")
    try:
        embedding_model = config["embedding_model"]
        embeddings = get_openai_embeddings(personas, model=embedding_model)
    except Exception as error:
        print(f"API call failed: {error}")
        embedding_model = "all-MiniLM-L6-v2"
        embeddings = get_fallback_embeddings(personas)

    embeddings = np.array(embeddings, dtype=np.float32)
    print(f"Embedding shape: {embeddings.shape}")
    print(f"Embedding deduplication (threshold={EMBEDDING_THRESHOLD})...")

    n = len(personas)
    keep_indices = []
    removed_pairs = []
    removed_flags = np.zeros(n, dtype=bool)

    # Compare each kept item with the items after it
    for i in range(n):
        if removed_flags[i]:
            continue

        keep_indices.append(i)

        if i + 1 < n:
            current_embedding = embeddings[i:i+1]
            remaining_embeddings = embeddings[i+1:]
            similarities = cosine_similarity(current_embedding, remaining_embeddings)[0]
            similar_indices = np.where(similarities > EMBEDDING_THRESHOLD)[0]

            for rel_idx in similar_indices:
                abs_idx = i + 1 + rel_idx
                if not removed_flags[abs_idx]:
                    removed_flags[abs_idx] = True
                    removed_pairs.append({
                        "keep_idx": i,
                        "remove_idx": abs_idx,
                        "similarity": similarities[rel_idx],
                        "keep_text": personas[i],
                        "remove_text": personas[abs_idx],
                    })

    final_personas = [personas[i] for i in keep_indices]

    print("Saving the deduplicated personas...")
    with open(persona_final, "w", encoding="utf-8") as f:
        for persona in final_personas:
            f.write(persona + "\n")

    print("Writing the deduplication report...")
    with open(embedding_report, "w", encoding="utf-8") as f:
        f.write("Embedding deduplication report (paper method)\n")
        f.write("=" * 60 + "\n")
        f.write(f"Model: {embedding_model}\n")
        f.write(f"Threshold: {EMBEDDING_THRESHOLD}\n")
        f.write(f"Input count: {n}\n")
        f.write(f"Kept: {len(final_personas)}\n")
        f.write(f"Removed: {len(removed_pairs)}\n")
        if n > 0:
            f.write(f"Removal rate at this stage: {len(removed_pairs) / n * 100:.2f}%\n")
        f.write("\n" + "=" * 60 + "\n\n")

        if removed_pairs:
            # Highest similarity first
            removed_pairs.sort(key=lambda pair: pair["similarity"], reverse=True)
            f.write("Duplicate pairs:\n")
            f.write("-" * 60 + "\n\n")
            for idx, pair in enumerate(removed_pairs, 1):
                f.write(f"Duplicate pair {idx}:\n")
                f.write(f"Similarity: {pair['similarity']:.4f}\n")
                f.write(f"Kept (index {pair['keep_idx']}): {pair['keep_text']}\n")
                f.write(f"Removed (index {pair['remove_idx']}): {pair['remove_text']}\n")
                f.write("-" * 60 + "\n\n")
            similarities = [pair["similarity"] for pair in removed_pairs]
            f.write("Similarity summary:\n")
            f.write(f"Mean similarity: {np.mean(similarities):.4f}\n")
            f.write(f"Highest similarity: {np.max(similarities):.4f}\n")
            f.write(f"Lowest similarity: {np.min(similarities):.4f}\n")
        else:
            f.write("No pairs exceeded the threshold.\n")

    print("Embedding deduplication complete.")
    print(f"Model: {embedding_model}")
    print(f"Input count: {n}")
    print(f"Kept: {len(final_personas)}")
    print(f"Removed: {len(removed_pairs)}")
    print(f"Removal rate at this stage: {len(removed_pairs) / n * 100:.2f}%")
    print(f"Saved results to {persona_final}")
    print(f"Saved report to {embedding_report}")

deduplicate_by_embedding()
